# 05 · Selección del ganador, prueba única y Model Registry

Para cada horizonte:
1. Ganador y segundo mejor según la CV del tuning (media ± desviación y diferencia pareada por pliegue).
2. Ambos se evalúan **una sola vez** en el hold-out (últimos 6 meses), nunca visto hasta aquí.
3. Métrica de negocio: días de alta demanda **no anticipados** (FN) frente a la regla del PSet 1,
   con un umbral elegido en la CV (no en prueba): el de mayor recall cuya precisión es al menos la
   de la regla de negocio.
4. Costo (tiempo de entrenamiento/inferencia) y explicabilidad (importancia por permutación).
5. Registro: `ecu911-t{h}-modelo` con alias `champion` (ganador) y `challenger` (segundo).

In [ ]:
import os, sys, json, time, tempfile, pathlib
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1"); os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "true")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow import MlflowClient
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, precision_recall_curve

from src.datos import cargar_obt, construir_tabla_modelado, HORIZONTES, COLUMNAS_X, SEED, RAIZ
from src.validacion import separar_holdout, pliegues_temporales, tabla_p90, etiquetar, p90_objetivo
from src.experimento import (configurar_mlflow, evaluar_regla, score_regla_negocio, umbral_negocio,
                             metricas_negocio, registrar_corrida)
from src.reporte import guardar_fig, guardar_tabla

SINTETICO = os.getenv("ECU911_SINTETICO") == "1"
sns.set_theme(style="whitegrid")
obt, meta = cargar_obt()
configurar_mlflow()
cliente = MlflowClient()
dir_res = RAIZ / ("data/dev/resultados" if SINTETICO else "docs/resultados")
tuning = pd.read_csv(dir_res / "tuning_resumen.csv")
tuning[["h", "modelo", "pr_auc_default", "pr_auc_tuned", "std_tuned", "mejora", "mejora_cuenta"]].round(4)

## 1. Ganador vs. segundo mejor en CV

In [ ]:
def oof_de(run_id):
    d = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="oof.parquet")
    return pd.read_parquet(d)

def pr_auc_por_pliegue(oof):
    return oof.groupby("fold").apply(lambda g: average_precision_score(g.y, g.score), include_groups=False)

seleccion, comparaciones = {}, []
for h in HORIZONTES:
    t = tuning[tuning.h == h].sort_values("pr_auc_tuned", ascending=False).reset_index(drop=True)
    g, s = t.iloc[0], t.iloc[1]
    fg, fs = pr_auc_por_pliegue(oof_de(g.run_id_padre)), pr_auc_por_pliegue(oof_de(s.run_id_padre))
    dif = fg - fs
    seleccion[h] = {"ganador": g, "segundo": s}
    comparaciones.append({"h": h, "ganador": g.modelo, "segundo": s.modelo,
                          "cv_ganador": f"{fg.mean():.4f} ± {fg.std(ddof=0):.4f}",
                          "cv_segundo": f"{fs.mean():.4f} ± {fs.std(ddof=0):.4f}",
                          "diferencia_media": dif.mean(), "std_diferencia_pareada": dif.std(ddof=0),
                          "pliegues_que_gana": int((dif > 0).sum()),
                          "supera_variacion_entre_pliegues": bool(dif.mean() > max(fg.std(ddof=0), fs.std(ddof=0)))})
comp = pd.DataFrame(comparaciones)
guardar_tabla(comp.round(5), "seleccion_cv")
comp.round(4)

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONTES), figsize=(12, 3.8))
for ax, h in zip(axes, HORIZONTES):
    for rol, m in (("ganador", "o-"), ("segundo", "s--")):
        f = pr_auc_por_pliegue(oof_de(seleccion[h][rol].run_id_padre))
        ax.plot(f.index, f.values, m, label=f"{rol}: {seleccion[h][rol].modelo}")
    ax.set(title=f"PR-AUC por pliegue (h = {h})", xlabel="pliegue", ylabel="PR-AUC"); ax.legend(fontsize=8)
plt.tight_layout(); guardar_fig(fig, "seleccion_pliegues"); plt.show()

## 2. Evaluación única en el hold-out + métrica de negocio

In [ ]:
finales, curvas, estratos_tabla = [], {}, []
vol = obt.groupby("cod_canton").n_emergencias.mean()
estrato = pd.qcut(vol.rank(method="first"), 4, labels=["Q1 bajo", "Q2", "Q3", "Q4 alto"])

for h in HORIZONTES:
    tabla = construir_tabla_modelado(obt, h)
    train, test = separar_holdout(tabla, h)
    pl = pliegues_temporales(train, h)
    p90 = tabla_p90(train)                       # P90 con TODO el entrenamiento
    y_test = etiquetar(test, p90)

    # Regla de negocio: precisión en CV (fija el piso de precisión) y resultado en prueba
    r_neg = evaluar_regla(score_regla_negocio, train, h, pl)
    neg_cv = metricas_negocio(r_neg.oof.y, r_neg.oof.score, 1e-12)        # alerta si score > 0
    s_neg_test = score_regla_negocio(test.prom_4_mismo_dia, p90_objetivo(test, p90))
    neg_test = metricas_negocio(y_test, s_neg_test, 1e-12)
    curvas[(h, "Regla de negocio")] = (y_test, s_neg_test)

    for rol in ("ganador", "segundo"):
        fila = seleccion[h][rol]
        pipe = mlflow.sklearn.load_model(f"runs:/{fila.run_id_padre}/modelo")   # entrenado con todo train
        oof = oof_de(fila.run_id_padre)
        umbral = umbral_negocio(oof.y.to_numpy(), oof.score.to_numpy(), precision_min=neg_cv["precision"])
        t0 = time.perf_counter()
        s_test = pipe.predict_proba(test[COLUMNAS_X])[:, 1]
        seg_inf = time.perf_counter() - t0
        pr_test = average_precision_score(y_test, s_test)
        m = metricas_negocio(y_test, s_test, umbral)
        m70 = metricas_negocio(y_test, s_test, 0.70)
        reduccion_fn = (neg_test["fn"] - m["fn"]) / neg_test["fn"]
        curvas[(h, fila.modelo)] = (y_test, s_test)
        for e in estrato.cat.categories:
            idx = test.cod_canton.map(estrato).eq(e).to_numpy()
            estratos_tabla.append({"h": h, "modelo": fila.modelo, "estrato": e, "prevalencia": y_test[idx].mean(),
                                   "pr_auc_test": average_precision_score(y_test[idx], s_test[idx])})
        run_final = registrar_corrida(
            f"final_{rol}_{fila.modelo}_t{h}", "final", fila.familia, h, json.loads(fila.mejores_params), None, meta,
            pipeline=pipe, X_ejemplo=train[COLUMNAS_X].sample(200, random_state=SEED),
            metricas_extra={"pr_auc_test": pr_test, "pr_auc_cv_mean": fila.pr_auc_tuned, "pr_auc_cv_std": fila.std_tuned,
                            "umbral_negocio": umbral, "precision_min_negocio": neg_cv["precision"],
                            "test_tp": m["tp"], "test_fp": m["fp"], "test_fn": m["fn"], "test_precision": m["precision"],
                            "test_recall": m["recall"], "test_alertas": m["alertas"],
                            "baseline_negocio_test_fn": neg_test["fn"], "baseline_negocio_test_precision": neg_test["precision"],
                            "reduccion_fn_vs_negocio": reduccion_fn, "test_recall_umbral_070": m70["recall"],
                            "test_precision_umbral_070": m70["precision"],
                            "segundos_inferencia_test": seg_inf, "segundos_entrenamiento": fila.segundos_entrenamiento / 5},
            tags_extra={"rol": "champion" if rol == "ganador" else "challenger", "run_tuning": fila.run_id_padre})
        finales.append({"h": h, "rol": rol, "modelo": fila.modelo, "run_id_final": run_final,
                        "pr_auc_cv": fila.pr_auc_tuned, "pr_auc_test": pr_test, "umbral": umbral,
                        "precision_test": m["precision"], "recall_test": m["recall"], "fn_test": m["fn"],
                        "alertas_test": m["alertas"], "fn_regla_negocio": neg_test["fn"],
                        "precision_regla_negocio": neg_test["precision"], "recall_regla_negocio": neg_test["recall"],
                        "reduccion_fn_vs_negocio": reduccion_fn, "recall_umbral_070": m70["recall"],
                        "precision_umbral_070": m70["precision"],
                        "seg_entrenamiento_por_ajuste": fila.segundos_entrenamiento / 5,
                        "ms_inferencia_por_1000_filas": seg_inf / len(test) * 1e6})
    print(f"h={h}: positivos en prueba {y_test.sum():,} de {len(y_test):,} ({y_test.mean():.2%}); "
          f"regla de negocio en prueba: precisión {neg_test['precision']:.3f}, recall {neg_test['recall']:.3f}, FN {neg_test['fn']:,}")

finales = pd.DataFrame(finales)
guardar_tabla(finales.round(5), "seleccion_final")
finales.drop(columns="run_id_final").round(4)

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONTES), figsize=(13, 4.5))
for ax, h in zip(axes, HORIZONTES):
    for (hh, nombre), (y, s) in curvas.items():
        if hh != h: continue
        p, r, _ = precision_recall_curve(y, s)
        ax.plot(r, p, label=f"{nombre} (PR-AUC {average_precision_score(y, s):.3f})")
    ax.axhline(y.mean(), ls=":", c="gray", label="azar")
    for f in finales[finales.h == h].itertuples():
        ax.plot(f.recall_test, f.precision_test, "k*", ms=12)
    ax.set(title=f"Curva precisión-recall en el hold-out (h = {h})", xlabel="recall", ylabel="precisión")
    ax.legend(fontsize=8)
plt.tight_layout(); guardar_fig(fig, "seleccion_curvas_pr"); plt.show()

In [ ]:
est = pd.DataFrame(estratos_tabla)
guardar_tabla(est.round(4), "seleccion_estratos")
est.pivot_table(index=["h", "modelo"], columns="estrato", values="pr_auc_test", observed=True).round(3)

## 3. Explicabilidad: importancia por permutación (ganador, hold-out)

In [ ]:
imp_tabla = []
for h in HORIZONTES:
    tabla = construir_tabla_modelado(obt, h)
    train, test = separar_holdout(tabla, h)
    y_test = etiquetar(test, tabla_p90(train))
    fila = seleccion[h]["ganador"]
    pipe = mlflow.sklearn.load_model(f"runs:/{fila.run_id_padre}/modelo")
    muestra = test.sample(min(20_000, len(test)), random_state=SEED)
    ym = etiquetar(muestra, tabla_p90(train))
    pi = permutation_importance(pipe, muestra[COLUMNAS_X], ym, scoring="average_precision",
                                n_repeats=3, random_state=SEED, n_jobs=1)
    imp_tabla.append(pd.DataFrame({"h": h, "modelo": fila.modelo, "columna": COLUMNAS_X,
                                   "importancia": pi.importances_mean, "std": pi.importances_std}))
imp = pd.concat(imp_tabla, ignore_index=True).sort_values(["h", "importancia"], ascending=[True, False]).reset_index(drop=True)
guardar_tabla(imp.round(5), "seleccion_importancia")
g = sns.catplot(data=imp.groupby("h").head(12).reset_index(drop=True), x="importancia", y="columna", col="h", kind="bar",
                sharey=False, height=4, aspect=1.2)
g.set_axis_labels("caída de PR-AUC al permutar", "")
guardar_fig(g.figure, "seleccion_importancia"); plt.show()

## 4. Registro en el Model Registry

In [ ]:
registro = []
for h in HORIZONTES:
    nombre = f"{'dev-' if SINTETICO else ''}ecu911-t{h}-modelo"
    for rol, alias in (("ganador", "champion"), ("segundo", "challenger")):
        run_id = finales[(finales.h == h) & (finales.rol == rol)].run_id_final.iloc[0]
        v = mlflow.register_model(f"runs:/{run_id}/modelo", nombre)
        cliente.set_registered_model_alias(nombre, alias, v.version)
        cliente.set_model_version_tag(nombre, v.version, "horizonte", f"t{h}")
        registro.append({"modelo_registrado": nombre, "alias": alias, "version": v.version,
                         "run_id": run_id, "modelo": seleccion[h][rol].modelo})
registro = pd.DataFrame(registro)
guardar_tabla(registro, "registro")
registro

DECISION_PLACEHOLDER